# VisionAI Inspector — Stage 2: Classical ML Baseline (Anomaly Detection)

**Goal:** establish a baseline using classical Scikit-learn models, before
touching deep learning in Stage 3.

**Protocol chosen (see Stage 1, section 11's open question):** anomaly
detection, faithful to MVTec AD's original design — we train only on normal
(`good`) images, and evaluate on the full, untouched `test` split (which
mixes normal and every defect type). We are **not** training a classifier
on labeled defect examples; we're teaching a model what "normal" looks like
and measuring how well it flags deviations at test time. This keeps results
comparable to the standard MVTec protocol and avoids ever training on
official test images.

Models: **One-Class SVM** and **Isolation Forest** — both trained on
normal-only data, both classical (no deep learning), both genuinely
different approaches to the same "what does normal look like" problem,
which makes comparing them instructive.

Like Stage 1, this notebook auto-detects whether real MVTec data is present
and falls back to a synthetic placeholder if not, with every such output
tagged **[SYNTHETIC PLACEHOLDER DATA]**.


## 1. Setup, imports, and loading Stage 1's splits

We load the exact train/val/test split Stage 1 produced. If those CSVs don't exist yet (e.g. you're running this notebook standalone), we regenerate them here using the identical protocol-preserving logic from `src/data_utils.build_protocol_splits` — but running Stage 1's notebook first is the normal path.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_utils import build_protocol_splits, discover_categories, make_synthetic_placeholder_dataset
from features import build_feature_matrix, feature_names

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "mvtec_ad"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CATEGORY = "bottle"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

category_path = RAW_ROOT / CATEGORY
USING_SYNTHETIC_DATA = not category_path.exists()
if USING_SYNTHETIC_DATA:
    print("Real MVTec data not found — generating synthetic placeholder for pipeline testing.")
    make_synthetic_placeholder_dataset(RAW_ROOT, category=CATEGORY, seed=RANDOM_SEED)

DATA_TAG = "[SYNTHETIC PLACEHOLDER DATA] " if USING_SYNTHETIC_DATA else ""

train_csv = PROCESSED_ROOT / f"{CATEGORY}_train.csv"
val_csv = PROCESSED_ROOT / f"{CATEGORY}_val.csv"
test_csv = PROCESSED_ROOT / f"{CATEGORY}_test.csv"

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    print("Loading splits saved by Stage 1 notebook...")
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
else:
    print("Stage 1 CSVs not found — regenerating splits with the same protocol-preserving logic.")
    train_df, val_df, test_df = build_protocol_splits(category_path, val_frac=0.15, seed=RANDOM_SEED)

print(f"\n{DATA_TAG}train (normal only): {len(train_df)}")
print(f"{DATA_TAG}val (normal only):   {len(val_df)}")
print(f"{DATA_TAG}test (untouched):    {len(test_df)}  -- label counts: {test_df['label'].value_counts().to_dict()}")

assert (train_df["defect_type"] == "good").all(), "train must be normal-only"
assert (val_df["defect_type"] == "good").all(), "val must be normal-only"


## 2. Feature extraction

We turn each image into a small hand-crafted feature vector using
`extract_grid_features` (see `src/features.py` for the full reasoning): a
4x4 grid over a grayscale, resized copy of the image, with per-cell mean
intensity, std of intensity, and mean gradient magnitude, plus two
whole-image statistics — 50 numbers per image total.

This is deliberately simple and interpretable. It won't compete with a CNN
(Stage 3), but that's the point of a baseline: a cheap, explainable
reference point everything later has to beat.


In [ ]:
GRID_SIZE = 4
TARGET_SIZE = (128, 128)

print("Extracting features for train/val/test (this reads every image once)...")
X_train = build_feature_matrix(train_df["filepath"].tolist(), target_size=TARGET_SIZE, grid_size=GRID_SIZE)
X_val = build_feature_matrix(val_df["filepath"].tolist(), target_size=TARGET_SIZE, grid_size=GRID_SIZE)
X_test = build_feature_matrix(test_df["filepath"].tolist(), target_size=TARGET_SIZE, grid_size=GRID_SIZE)

y_test_true = (test_df["label"] == "defective").astype(int).to_numpy()  # 1 = defective, 0 = good

print(f"{DATA_TAG}X_train shape: {X_train.shape}")
print(f"{DATA_TAG}X_val shape:   {X_val.shape}")
print(f"{DATA_TAG}X_test shape:  {X_test.shape}")
print(f"{DATA_TAG}Feature names (first 6 of {len(feature_names(GRID_SIZE))}): {feature_names(GRID_SIZE)[:6]}")

## 3. Feature scaling — fit on train only

Same leakage principle as Stage 1's normalization: any statistic used to
transform the data (here, per-feature mean/std for standardization) must be
computed **only** from training data, then applied unchanged to val/test.
Fitting the scaler on val or test would leak their distribution into a
number the model effectively uses.

Scaling matters specifically for One-Class SVM: it's a distance/kernel-based
method (RBF kernel), and features on very different scales (e.g. a `mean`
feature in the 0-255 range vs a `std` feature typically much smaller) would
let the larger-scale features dominate distance calculations regardless of
which is actually more informative.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"{DATA_TAG}Post-scaling train feature mean (~0 expected): {X_train_scaled.mean():.4f}")
print(f"{DATA_TAG}Post-scaling train feature std (~1 expected):  {X_train_scaled.std():.4f}")

## 4. Model 1 — One-Class SVM

**What problem it solves:** learns a boundary around the "normal" region of
feature space, using only normal examples. At inference, points outside that
boundary are flagged as anomalies.

**Key parameters:**
- `kernel="rbf"` — allows a flexible, non-linear boundary rather than a
  straight hyperplane, since "normal bottle" isn't necessarily a simple
  linear region of feature space.
- `nu` — simultaneously an upper bound on the fraction of training points
  allowed to be margin errors, and a lower bound on the fraction of support
  vectors. In practice: a rough prior on what fraction of our own "normal"
  training data we expect to be borderline/noisy. We set it modestly low
  (0.05) since our training images are all genuinely normal by protocol,
  but real photos always have some natural variation.
- `gamma="scale"` — controls how far the influence of a single training
  point reaches; `"scale"` picks a sensible default from the data's
  variance rather than requiring manual tuning.

**How we turn its output into a decision:** `decision_function` gives a
continuous score (higher = more normal). Rather than trusting the model's
internal default threshold blindly, we calibrate a threshold using the
**validation set** (still normal-only) — the score below which we'd
incorrectly flag 5% of genuinely normal validation images. We deliberately
never look at test labels to pick this threshold.


In [ ]:
ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm.fit(X_train_scaled)

val_scores_ocsvm = ocsvm.decision_function(X_val_scaled)
test_scores_ocsvm = ocsvm.decision_function(X_test_scaled)

# Threshold calibrated ONLY on normal validation scores: the 5th percentile
# means we accept ~95% of genuinely normal validation images as "normal".
THRESHOLD_PERCENTILE = 5
ocsvm_threshold = np.percentile(val_scores_ocsvm, THRESHOLD_PERCENTILE)
print(f"{DATA_TAG}One-Class SVM threshold (from val, {THRESHOLD_PERCENTILE}th percentile): {ocsvm_threshold:.4f}")

y_pred_ocsvm = (test_scores_ocsvm < ocsvm_threshold).astype(int)  # 1 = flagged defective

## 5. Model 2 — Isolation Forest

**What problem it solves:** a completely different mechanism from OCSVM.
It builds an ensemble of random trees that repeatedly split the feature
space on random features/thresholds. Anomalies are points that get
isolated (end up alone in a leaf) after very few random splits, because
they sit in sparse, unusual regions of feature space — normal points,
clustered densely together, take many more splits to isolate. The average
path length across all trees becomes the anomaly score.

**Key parameters:**
- `n_estimators` — number of random trees; more trees give a more stable
  average path length estimate.
- `contamination` — scikit-learn uses this to set its internal default
  threshold. Since our training data is normal-only by protocol, we set
  it low (0.01) to reflect that we don't expect meaningful contamination —
  but as with OCSVM, we calibrate our actual operating threshold from
  validation scores rather than relying on this parameter alone.

Same validation-based threshold calibration as OCSVM, for a fair
comparison between the two models.


In [ ]:
iso_forest = IsolationForest(n_estimators=200, contamination=0.01, random_state=RANDOM_SEED)
iso_forest.fit(X_train_scaled)

val_scores_iso = iso_forest.decision_function(X_val_scaled)   # higher = more normal
test_scores_iso = iso_forest.decision_function(X_test_scaled)

iso_threshold = np.percentile(val_scores_iso, THRESHOLD_PERCENTILE)
print(f"{DATA_TAG}Isolation Forest threshold (from val, {THRESHOLD_PERCENTILE}th percentile): {iso_threshold:.4f}")

y_pred_iso = (test_scores_iso < iso_threshold).astype(int)  # 1 = flagged defective

## 6. Evaluation on the untouched test set

This is the first and only time test labels are used — purely to measure
performance, never to influence training or thresholding above.

We report:
- **Confusion matrix** — counts of correct/incorrect good vs defective calls.
- **Precision, recall, F1** (via `classification_report`) — since "defective"
  is the minority, positive class, recall (catching real defects) and
  precision (not crying wolf on normal parts) trade off against each other
  based on where the threshold sits.
- **ROC-AUC** — a threshold-independent summary of how well the continuous
  anomaly score separates good from defective across all possible
  thresholds, useful for comparing the two models independent of our
  specific percentile choice above.


In [ ]:
def evaluate(name, y_true, y_pred, scores, higher_score_is_normal=True):
    print(f"=== {DATA_TAG}{name} ===")
    print(classification_report(y_true, y_pred, target_names=["good", "defective"]))
    auc_scores = scores if higher_score_is_normal else -scores
    auc = roc_auc_score(y_true, -auc_scores)  # roc_auc_score wants higher score = positive class (defective)
    print(f"ROC-AUC: {auc:.4f}")
    return auc

auc_ocsvm = evaluate("One-Class SVM", y_test_true, y_pred_ocsvm, test_scores_ocsvm)
print()
auc_iso = evaluate("Isolation Forest", y_test_true, y_pred_iso, test_scores_iso)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay.from_predictions(y_test_true, y_pred_ocsvm, display_labels=["good", "defective"], ax=axes[0], colorbar=False)
axes[0].set_title(f"{DATA_TAG}One-Class SVM")

ConfusionMatrixDisplay.from_predictions(y_test_true, y_pred_iso, display_labels=["good", "defective"], ax=axes[1], colorbar=False)
axes[1].set_title(f"{DATA_TAG}Isolation Forest")

plt.tight_layout()
plt.show()

## 7. ROC curves and score-separation visualization

The ROC curve shows the full precision/recall tradeoff across every possible threshold, not just the one we picked. The score histograms show *why* a model does or doesn't separate well: good separation means the good/defective score distributions barely overlap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr_ocsvm, tpr_ocsvm, _ = roc_curve(y_test_true, -test_scores_ocsvm)
fpr_iso, tpr_iso, _ = roc_curve(y_test_true, -test_scores_iso)

axes[0].plot(fpr_ocsvm, tpr_ocsvm, label=f"One-Class SVM (AUC={auc_ocsvm:.3f})")
axes[0].plot(fpr_iso, tpr_iso, label=f"Isolation Forest (AUC={auc_iso:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="random chance")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title(f"{DATA_TAG}ROC curves")
axes[0].legend()

good_mask = y_test_true == 0
axes[1].hist(test_scores_ocsvm[good_mask], bins=15, alpha=0.6, label="good (true)", color="#4C72B0")
axes[1].hist(test_scores_ocsvm[~good_mask], bins=15, alpha=0.6, label="defective (true)", color="#C44E52")
axes[1].axvline(ocsvm_threshold, color="black", linestyle="--", label="threshold (from val)")
axes[1].set_title(f"{DATA_TAG}One-Class SVM score separation")
axes[1].set_xlabel("decision_function score (higher = more normal)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Model comparison summary

In [ ]:
comparison = pd.DataFrame({
    "model": ["One-Class SVM", "Isolation Forest"],
    "roc_auc": [auc_ocsvm, auc_iso],
    "threshold_percentile_used": [THRESHOLD_PERCENTILE, THRESHOLD_PERCENTILE],
})
print(f"{DATA_TAG}")
comparison

## 9. Save Stage 2 outputs

Persist predictions and scores so Stage 3 (deep learning) has a fixed baseline number to compare against, without needing to re-run this notebook.

In [ ]:
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

results_df = test_df[["filepath", "label", "defect_type"]].copy()
results_df["ocsvm_score"] = test_scores_ocsvm
results_df["ocsvm_pred_defective"] = y_pred_ocsvm
results_df["isoforest_score"] = test_scores_iso
results_df["isoforest_pred_defective"] = y_pred_iso

out_path = PROCESSED_ROOT / f"{CATEGORY}_stage2_baseline_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved {len(results_df)} rows -> {out_path}")

summary_path = PROCESSED_ROOT / f"{CATEGORY}_stage2_baseline_summary.csv"
comparison.to_csv(summary_path, index=False)
print(f"Saved model comparison -> {summary_path}")

print(f"\n{'SYNTHETIC PLACEHOLDER outputs saved.' if USING_SYNTHETIC_DATA else 'Real dataset outputs saved.'}")

## 10. Summary — Stage 2 complete

What we built:

- A hand-crafted, documented feature extraction pipeline (`src/features.py`)
  turning each image into 50 interpretable numbers.
- A leakage-aware scaling step (fit on train only).
- Two classical anomaly-detection models — **One-Class SVM** and
  **Isolation Forest** — both trained on normal-only data, faithful to
  MVTec AD's original protocol.
- A validation-calibrated decision threshold for each model, chosen without
  ever looking at test labels.
- Evaluation on the untouched test set: confusion matrix, precision/recall/
  F1, ROC-AUC, and ROC curve comparison.
- Saved baseline predictions and a model comparison table, for Stage 3 to
  beat.

Reminder: if outputs above are tagged **[SYNTHETIC PLACEHOLDER DATA]**, the
ROC-AUC numbers are meaningless (random noise images have no real defect
structure to detect) — they only confirm the code runs. Re-run against the
real `bottle` dataset (with Stage 1's section 3b showing all `OK`) before
treating any number here as a real result.

**Next (Stage 3):** build a CNN-based approach in PyTorch and compare its
performance against this baseline on the exact same untouched test set.
